In [1]:
from delta.tables import DeltaTable
from pyspark.sql.functions import col

# 1. Pobieramy nowe dane z Bronze
# Tutaj zamieniamy Stringi z API na Double dla GPS
new_gps_data = spark.table("bronze_gps").select(
    col("VehicleNumber").alias("bus_id"),
    col("Lines").alias("line"),
    col("Lat").cast("double").alias("latitude"),
    col("Lon").cast("double").alias("longitude"),
    col("Time").alias("signal_time"),
    col("ingestion_time")
).dropDuplicates(["bus_id", "signal_time"])

# 2. Logika MERGE (Upsert)
# Jeśli tabela Silver nie istnieje - stwórz ją
if not spark.catalog.tableExists("silver_bus_current"):
    new_gps_data.write.format("delta").saveAsTable("silver_bus_current")
    print("Utworzono tabelę Silver.")
else:
    # Jeśli istnieje - zaktualizuj istniejące autobusy lub dodaj nowe
    target_table = DeltaTable.forName(spark, "silver_bus_current")
    
    (target_table.alias("t")
        .merge(new_gps_data.alias("s"), "t.bus_id = s.bus_id")
        .whenMatchedUpdate(condition="t.signal_time < s.signal_time", set={
            "latitude": "s.latitude",
            "longitude": "s.longitude",
            "signal_time": "s.signal_time",
            "ingestion_time": "s.ingestion_time"
        })
        .whenNotMatchedInsertAll()
        .execute())
    print("Zaktualizowano tabelę Silver (Merge zakończony).")


StatementMeta(, 2d6bc8a6-9f1d-4d06-a496-83af7c2499d4, 3, Finished, Available, Finished, False)

Utworzono tabelę Silver.
